# Вариант 03 — анализ продаж супермаркета

Открой этот ноутбук рядом с двумя исходными CSV и нажми **Run All**. После запуска в папке с данными появится `cleaned_data.csv`. Текстовые выводы написаны для исходных CSV варианта 03. Перед защитой прочитай их вместе с таблицами и графиками.

## 1. Первичный анализ

Проверяем размеры таблиц, типы данных, полные дубликаты, пропуски и варианты написания категорий и способов оплаты.



In [ ]:
# Загружаем оба CSV и проверяем исходные данные.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

prodazhi_raw = pd.read_csv("03_supermarket_raw.csv")
spravochnik_raw = pd.read_csv("03_supermarket_lookup.csv")

print("Razmer prodazh:", prodazhi_raw.shape)
prodazhi_raw.info()
display(prodazhi_raw.head(10))

display(spravochnik_raw.head(10))

print("Polnye dublikaty:", prodazhi_raw.duplicated().sum())

display(prodazhi_raw[prodazhi_raw.duplicated(keep=False)].head(6))

display(prodazhi_raw.isna().sum())
display(spravochnik_raw.isna().sum())

chislovye = prodazhi_raw.select_dtypes(include="number").columns
ostalnye = prodazhi_raw.columns.difference(chislovye)
print("Chislovye:", chislovye.tolist())
print("Ostalnye:", ostalnye.tolist())

for stolbec in ["Product Category", "Payment Method"]:
    print(stolbec, prodazhi_raw[stolbec].unique())

В таблице продаж 840 строк и 12 столбцов. Повторяются 19 полных строк. В категориях встречаются пробелы и разный регистр. Количество хранится как текст, даты записаны в двух форматах, а некоторые цены значительно выше остальных.

## 2. Очистка и объединение

Сравниваем полные исходные строки при удалении дубликатов. Для ключа объединения в обеих таблицах убираем пробелы по краям, унифицируем регистр и превращаем пустые строки в пропуски. По предварительной проверке все заполненные категории совпали после этой нормализации, поэтому дополнительный словарь синонимов не требуется. Неизвестные категории оставляем пустыми. Способы оплаты также приводим к одному регистру, чтобы одинаковые способы учитывались вместе.

In [ ]:
# Удаляем дубликаты и приводим категории, количество и даты к нужному формату.
prodazhi = prodazhi_raw.drop_duplicates().copy()
spravochnik = spravochnik_raw.copy()

for tablica in [prodazhi, spravochnik]:
    kategorii = tablica["Product Category"].astype("string")
    kategorii = kategorii.str.strip().str.lower()
    tablica["Product Category"] = kategorii.replace({"": pd.NA, "n/a": pd.NA})

oplata_tekst = prodazhi["Payment Method"].astype("string")
oplata_tekst = oplata_tekst.str.strip().str.lower()
prodazhi["Payment Method"] = oplata_tekst.replace({"": pd.NA, "n/a": pd.NA})

tekst_kolichestva = prodazhi["units_sold_raw"].astype("string").str.strip()
proverka_kolichestva = pd.to_numeric(tekst_kolichestva, errors="coerce")
maska_oshibok = tekst_kolichestva.notna() & proverka_kolichestva.isna()
display(tekst_kolichestva[maska_oshibok].value_counts())
tekst_kolichestva = tekst_kolichestva.mask(maska_oshibok)
prodazhi["units_sold_raw"] = tekst_kolichestva.astype("Float64")

tekst_daty = prodazhi["Sale Date"].astype("string").str.strip()
daty_iso = pd.to_datetime(tekst_daty, format="%Y-%m-%d", errors="coerce")
daty_s_tochkami = pd.to_datetime(tekst_daty, format="%d.%m.%Y", errors="coerce")
prodazhi["Sale Date"] = daty_iso.fillna(daty_s_tochkami)
display(tekst_daty[tekst_daty.notna() & prodazhi["Sale Date"].isna()].value_counts())

print("Udaleno dublikatov:", len(prodazhi_raw) - len(prodazhi))
print("Strok bez kategorii:", prodazhi["Product Category"].isna().sum())
print("Propuski v kolichestve:", prodazhi["units_sold_raw"].isna().sum())
print("Propuski v datah:", prodazhi["Sale Date"].isna().sum())

Количество приводим через `astype("Float64")`, чтобы сохранить возможные пропуски. Даты разбираем отдельно по двум форматам. Для цен применяем порог 1,5 межквартильного размаха. Цены за пределами отмечаем как пропущенные для последующих расчётов и сохраняем сами строки. Это аналитическое допущение. IQR выявляет необычные цены, а подтвердить ошибку можно по первичным записям.

In [ ]:
# Находим выбросы по IQR и сравниваем максимальную цену до и после очистки.
ceny = pd.to_numeric(prodazhi["Unit Price"], errors="coerce")
pervyi_kvartil = ceny.quantile(0.25)
tretii_kvartil = ceny.quantile(0.75)
razmah = tretii_kvartil - pervyi_kvartil
nizhnyaya_granica = pervyi_kvartil - 1.5 * razmah
verhnyaya_granica = tretii_kvartil + 1.5 * razmah
maska_vybrosov = (ceny < nizhnyaya_granica) | (ceny > verhnyaya_granica)
display(prodazhi.loc[maska_vybrosov, ["Product Category", "Store Branch", "Unit Price"]]
        .sort_values("Unit Price", ascending=False))

prodazhi["Unit Price"] = ceny.mask(maska_vybrosov)

print("Granicy:", nizhnyaya_granica, verhnyaya_granica)
print("Vybrosy:", maska_vybrosov.sum())
print("Maksimum do:", ceny.max())
print("Maksimum posle:", prodazhi["Unit Price"].max())

Границы IQR получились −20 344,50 и 74 075,50 сома. За границами 10 цен. Максимум изменился с 1 736 400 до 49 930 сомов. Отмеченные цены оставили пустыми, сохранив строки.

Используем левое объединение, чтобы сохранить записи о продажах даже при отсутствии категории в справочнике. Для каждой конфликтующей категории выбираем первую запись в порядке исходного справочника. В этом CSV первыми идут Category A–D. Показываем конфликтующие строки, поскольку выбор определяет сегмент и приоритет. Затем проверяем уникальность ключа, чтобы соединение не увеличило число продаж.

In [ ]:
# Объединяем продажи со справочником и сохраняем очищенный CSV.
klyuch = "Product Category"
zapisi_s_klyuchom = spravochnik.dropna(subset=[klyuch]).drop_duplicates()
maska_konfliktov = zapisi_s_klyuchom.duplicated(subset=[klyuch], keep=False)
print("Konfliktuyushchie zapisi spravochnika:")
display(zapisi_s_klyuchom.loc[maska_konfliktov].sort_values(klyuch))

spravochnik_s_klyuchom = zapisi_s_klyuchom.drop_duplicates(subset=[klyuch], keep="first")
dannye = prodazhi.merge(
    spravochnik_s_klyuchom, on="Product Category", how="left",
    validate="many_to_one", indicator=True, suffixes=("", "_spravochnik"),
)
bez_sovpadeniya = dannye.loc[dannye["_merge"].eq("left_only"), "Product Category"]
print("Strok do i posle merge:", len(prodazhi), len(dannye))
print("Strok bez sovpadeniya:", len(bez_sovpadeniya))
display(bez_sovpadeniya.value_counts(dropna=False))

dannye = dannye.drop(columns="_merge")
imena = dannye.columns.str.strip().str.lower()
dannye.columns = imena.str.replace(r"\s+", "_", regex=True)
dannye = dannye.rename(columns={"units_sold_raw": "units_sold"})
pervye_stolbcy = ["sale_date", "store_branch", "product_category", "payment_method",
                  "unit_price", "units_sold"]
ostalnye_stolbcy = [stolbec for stolbec in dannye.columns if stolbec not in pervye_stolbcy]
dannye = dannye[pervye_stolbcy + ostalnye_stolbcy]

dannye.info()
display(dannye.isna().sum().rename("propuski_posle_ochistki"))
print("Polnye dublikaty posle ochistki:", dannye.duplicated().sum())
dannye.to_csv("cleaned_data.csv", index=False)

## 3. NumPy

### 3.1–3.3. Массивы, индексация и векторизация

Для деления берём строки с известной ценой и положительным количеством. Из этих же строк извлекаем оба массива, чтобы пары «цена — количество» соответствовали одной записи. Массив NumPy имеет общий тип элементов и поддерживает вычисления сразу над всеми элементами. Список Python может содержать разные типы, а его умножение на целое число повторяет элементы.

Срез `[10:15]` возвращает пять элементов с индексами 10–14. `argmax` возвращает индекс первого максимума. В задании показатель NumPy задан как деление цены на количество. Это расчётное отношение в сомах на единицу количества. Стоимость покупки определяется отдельно как произведение цены и количества.

In [ ]:
# Создаём массивы NumPy и рассчитываем отношение цены к количеству.
maska = dannye["unit_price"].notna() & (dannye["units_sold"] > 0)
vyborka_numpy = dannye.loc[maska.fillna(False)]
massiv_cen = np.array(vyborka_numpy["unit_price"], dtype=float)
massiv_kolichestv = np.array(vyborka_numpy["units_sold"], dtype=float)

for nazvanie, massiv in [("Cena", massiv_cen), ("Kolichestvo", massiv_kolichestv)]:
    print(nazvanie, "shape:", massiv.shape, "ndim:", massiv.ndim,
          "size:", massiv.size, "dtype:", massiv.dtype)

print("Vosem koefficientov:", np.linspace(0.8, 1.2, 8))
print("Srez cen:", massiv_cen[10:15])
indeks_maksimuma = np.argmax(massiv_cen)
print("Indeks v massive:", indeks_maksimuma, "Maksimum:", massiv_cen[indeks_maksimuma])

pokazatel = massiv_cen / massiv_kolichestv
print("Pervaya zapis: cena, kolichestvo, otnoshenie:",
      massiv_cen[0], massiv_kolichestv[0], pokazatel[0])

### 3.4–3.7. Broadcasting, маска и агрегирование

Массив коэффициентов формы `(5,)` выравнивается как `(1, 5)`. При умножении на `(8, 1)` единичные измерения расширяются до нужных размеров. Каждая строка результата соответствует одной записи, каждый столбец — одному сценарию. При умножении массивов одинаковой формы элементы уже попарно выровнены.

`axis=0` сворачивает строки и даёт минимум для каждого сценария. `axis=1` сворачивает столбцы и даёт сумму сценариев для каждой записи. Сумма сценариев здесь демонстрирует операцию над осью. Накопительная сумма показывает, как меняется сумма первых десяти отношений при добавлении записей в порядке массива.

Векторизация снижает накладные расходы Python: операции над числовым массивом выполняются в скомпилированном коде NumPy без вызова тела Python-цикла для каждой записи.

In [ ]:
# Проверяем сценарии, отбираем записи и считаем итоги по осям.
koefficienty = np.array([0.80, 0.90, 1.00, 1.10, 1.25])
stolbec_znacheniy = pokazatel[:8].reshape(8, 1)
scenarii = stolbec_znacheniy * koefficienty
print("Forma scenariev:", scenarii.shape)
display(pd.DataFrame(scenarii, columns=koefficienty))

ne_menshe_treh = massiv_kolichestv >= 3
ne_bolshe_pyatnadcati = massiv_kolichestv <= 15
cena_nizhe_limita = massiv_cen < 8000
maska_otbora = ne_menshe_treh & ne_bolshe_pyatnadcati & cena_nizhe_limita
print("Podhodyashchie zapisi:", maska_otbora.sum())
print("Summa otnoshenii v otbore:", np.sum(pokazatel[maska_otbora]))
print("Minimumy po scenariyam:", scenarii.min(axis=0))
print("Summy po zapisyam:", scenarii.sum(axis=1))
print("Nakopitelnaya summa pervyh desyati:", np.cumsum(pokazatel[:10]))
print("Summa pervyh desyati:", np.sum(pokazatel[:10]))
print("Summa po vsem zapisyam:", np.sum(pokazatel))

Маске соответствуют 32 из 811 записей. Сумма расчётных отношений в этой группе — 20 930,73. Группу можно рассмотреть отдельно как покупки с заданным диапазоном количества и цены.

## 4. Анализ в pandas

Суммы ниже относятся к столбцу `unit_price` по условиям задания. Для выручки понадобилась бы отдельная агрегация стоимости покупок. Для сравнения категорий берём записи с известной категорией и пригодной ценой. В таблице оплаты используем все записи с известной категорией и способом оплаты, поскольку цена для этого расчёта не нужна.

In [ ]:
# Сравниваем категории, способы оплаты и филиалы с помощью pandas.
ceny_s_kategoriey = dannye.dropna(subset=["product_category", "unit_price"])
itogi_kategorii = (
    ceny_s_kategoriey.groupby("product_category")
    .agg({"unit_price": ["sum", "mean", "count", "max"]})["unit_price"]
)
display(itogi_kategorii.sort_values("sum", ascending=False))

oplata = pd.crosstab(
    dannye["product_category"], dannye["payment_method"], normalize="index"
).mul(100)
display(oplata.round(1))

ceny_s_filialom = ceny_s_kategoriey.dropna(subset=["store_branch"])
svodnaya = ceny_s_filialom.pivot_table(
    values="unit_price", index="store_branch", columns="product_category",
    aggfunc="sum", margins=True,
)
display(svodnaya)
bez_itogov = svodnaya.drop(index="All", columns="All")
yacheiki = bez_itogov.stack()
silnyi_filial, silnaya_kategoriya = yacheiki.idxmax()
print("Obshchii itog svodnoi:", svodnaya.loc["All", "All"])
print("Luchshaya para:", silnyi_filial, silnaya_kategoriya, yacheiki.max())

minimalnye_ceny = dannye.dropna(subset=["unit_price"]).nsmallest(5, "unit_price")
display(minimalnye_ceny)

s_rangom = ceny_s_kategoriey.assign(
    rang_ceny=ceny_s_kategoriey.groupby("product_category")["unit_price"]
    .rank(method="min", ascending=False)
)
top_tri = s_rangom.loc[s_rangom["rang_ceny"].le(3)]
display(top_tri.sort_values(["product_category", "rang_ceny"]))

У одинаковых цен один ранг. Поэтому при равенстве на границе в выдаче может оказаться больше трёх записей категории. Для пяти самых низких цен возможное объяснение — недорогой ассортимент или скидка. Эти причины нужно проверять по товарам и акциям. Таблица оплаты позволяет сравнить долю ведущего способа между категориями.

## Пять графиков Matplotlib

In [ ]:
# Показываем сумму цен по филиалам на столбчатой диаграмме.
itogi_filialov = dannye.dropna(subset=["store_branch", "unit_price"]).groupby("store_branch")["unit_price"].sum()
plt.figure(figsize=(9, 4))
plt.bar(itogi_filialov.index.astype(str), itogi_filialov)
plt.gca().set(title='Summa cen po filialam', xlabel='Filial', ylabel='Summa Unit Price, KGS')
plt.show()

print("Luchshii filial:", itogi_filialov.idxmax(), itogi_filialov.max())

Наибольшая сумма цен у Токмока — 5 154 862 сома. График включает записи с известной ценой, у которых категория может отсутствовать.

In [ ]:
# Показываем, как сумма цен менялась по месяцам.
po_mesyacam = (
    dannye.dropna(subset=["sale_date", "unit_price"])
    .groupby(pd.Grouper(key="sale_date", freq="MS"))["unit_price"].sum(min_count=1)
)
plt.figure(figsize=(9, 4))
plt.plot(po_mesyacam.index, po_mesyacam, marker="o")
plt.gca().set(title='Summa cen po mesyacam', xlabel='Mesyac', ylabel='Summa Unit Price, KGS')
plt.show()

print("Mesyac s maksimumom:", po_mesyacam.idxmax(), po_mesyacam.max())

Месячный максимум пришёлся на январь 2024 года — 1 109 434 сома.

In [ ]:
# Проверяем связь между количеством и ценой на диаграмме рассеяния.
plt.figure(figsize=(8, 4))
plt.scatter(massiv_kolichestv, massiv_cen, alpha=0.45)
plt.gca().set(title='Kolichestvo i cena', xlabel='Kolichestvo', ylabel='Unit Price, KGS')
plt.show()
korrelyaciya = vyborka_numpy["units_sold"].corr(vyborka_numpy["unit_price"])

print("Korrelyaciya:", round(korrelyaciya, 2))

Корреляция около −0,04. В этой выборке линейная связь между количеством и ценой слабая.

In [ ]:
# Показываем распределение цен на гистограмме.
ceny_dlya_grafika = dannye["unit_price"].dropna()
plt.figure(figsize=(8, 4))
plt.hist(ceny_dlya_grafika, bins=20, edgecolor="white")
plt.gca().set(title='Raspredelenie cen', xlabel='Unit Price, KGS', ylabel='Chislo zapisei')
plt.show()

print("Mediana:", ceny_dlya_grafika.median())
print("Srednee:", round(ceny_dlya_grafika.mean(), 2))
print("Asimmetriya:", round(ceny_dlya_grafika.skew(), 2))

Медиана — 26 265 сомов, среднее — 25 957,11 сома. Асимметрия −0,12 указывает на небольшой перекос.

In [ ]:
# Показываем долю каждой категории в общей сумме цен.
summy_kategorii = itogi_kategorii["sum"]
plt.figure(figsize=(8, 6))
plt.pie(summy_kategorii, labels=summy_kategorii.index, autopct="%1.1f%%")
plt.title("Doli kategorii v summe cen")
plt.show()
dolya_lidera = summy_kategorii.max() / summy_kategorii.sum() * 100

print("Dolya lidera, %:", round(dolya_lidera, 1))

На категорию A приходится 29,1% суммы цен среди записей с известной категорией.

## 5. Выводы и рекомендации

1. По сумме цен лидирует категория A (6 032 701 сом), самая низкая сумма у категории C (4 890 610 сомов). По средней цене лидирует категория D (26 682,50 сома).
2. По сумме цен лидирует Токмок (5 154 862 сома), самая низкая сумма у Оша (3 556 649 сомов). Самая сильная пара в сводной — Токмок и категория A (1 434 681 сом).
3. Максимальная цена после очистки — 49 930 сомов, категория A.

**Рекомендации.** Сравнить ассортимент и число продаж категории A в Токмоке с другими филиалами. Проверить 10 экстремальных цен по первичным записям.

## Вопросы для защиты

- Почему левое объединение сохраняет 13 записей без категории?
- Что случится с максимумом, если вернуть отмеченные IQR цены?
- Почему при умножении массивов `(8, 1)` и `(5,)` получается `(8, 5)`?
- Чем отличаются сумма цен, отношение цены к количеству и стоимость покупки?
- Как интерпретировать строку таблицы долей оплаты и почему её значения в сумме дают 100%?